# ForecastEx - focus on NYC Mayor group forecast

## Web REST API

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests, json, os
from pprint import pprint

## Get live markets

In [3]:
import requests
from extract_markets_with_categories import extract_markets_with_categories
from expand_cats import expand_cats

url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/forecasttrader/category/tree"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    json_data = response.json()
except Exception as e:
    print("Error during request:", e)

markets = [expand_cats(m) for m in extract_markets_with_categories(json_data['categories'])]
markets[0]

{'name': 'General Election for New York City Mayor',
 'symbol': 'MNYCG',
 'conid': 796056051,
 'category1': 'Elections',
 'category2': 'Mayoral',
 'category3': 'New York City'}

In [4]:
import pandas as pd
df_markets = pd.DataFrame(markets)

In [5]:
conid_to_market = {market['conid']: market for market in markets}

In [6]:
df_markets[df_markets.symbol == 'MNYCG']

,name,symbol,conid,category1,category2,category3
0,General Election for New York City Mayor,MNYCG,796056051,Elections,Mayoral,New York City


In [7]:
df_markets[(df_markets.category1 == 'Elections') & (df_markets.category2 == 'Mayoral')]

,name,symbol,conid,category1,category2,category3
0,General Election for New York City Mayor,MNYCG,796056051,Elections,Mayoral,New York City
1,Mayoral Democratic Primary in Seattle,MSEAD,771326556,Elections,Mayoral,Seattle
2,Mayoral General Election in Minneapolis,MMSPG,771326535,Elections,Mayoral,Minneapolis
3,Mayoral General Election in Atlanta,MATLG,771326527,Elections,Mayoral,Atlanta
4,Mayoral Primary Election in Detroit,MDETN,771326565,Elections,Mayoral,Detroit
5,Mayoral General Election in Miami,MMIAG,771326550,Elections,Mayoral,Miami
6,Mayoral General Election in Boston,MBOSG,771326590,Elections,Mayoral,Boston


In [8]:
market = markets[0]
market

{'name': 'General Election for New York City Mayor',
 'symbol': 'MNYCG',
 'conid': 796056051,
 'category1': 'Elections',
 'category2': 'Mayoral',
 'category3': 'New York City'}

## Get forecastex markets excluding financials

In [9]:
from fetch_all_contracts import fetch_all_contracts
from tqdm.auto import tqdm

In [10]:
for market in tqdm(markets):
    if 'contracts' not in market:
        market['contracts'] = fetch_all_contracts(market['conid'])
        print(market['conid'], len(market['contracts']))
    break

  0%|          | 0/222 [00:00<?, ?it/s]

796056051 10


In [11]:
forecastex = [market] #  for market in markets if len(market['contracts']) > 0]

In [12]:
len(markets), len(forecastex)

(222, 1)

## Get current probability for each market contract

In [13]:
from get_candidate_probability import get_candidate_probability

In [14]:
from tqdm.auto import tqdm

In [15]:
for market in tqdm(forecastex):
    for contract in market['contracts']:
        conid = contract['conid']
        if 'probability' not in contract:
            try:
                contract['probability'] = get_candidate_probability(conid)
            except:
                contract['probability'] = None
                print('error', contract['shortDescription'])
        try:
            print(conid, contract['shortDescription'], contract['probability']['probability_pct'])
        except:
            pass

  0%|          | 0/1 [00:00<?, ?it/s]

796056496 MNYCG Nov04'25 Sliwa YES @FORECASTX 4.0
796056501 MNYCG Nov04'25 Sliwa NO @FORECASTX 97.0
796056506 MNYCG Nov04'25 Cuomo YES @FORECASTX 12.0
796056511 MNYCG Nov04'25 Cuomo NO @FORECASTX 89.0
796056520 MNYCG Nov04'25 Mamdani YES @FORECASTX 77.0
796056525 MNYCG Nov04'25 Mamdani NO @FORECASTX 24.0
796056531 MNYCG Nov04'25 Adams YES @FORECASTX 10.0
796056534 MNYCG Nov04'25 Adams NO @FORECASTX 91.0


In [16]:
f2 = [x.copy() for x in forecastex]

In [17]:
for market in f2:
    market['contracts'] = [x for x in market['contracts'] if x['probability']]

In [18]:
f2 = [market for market in f2 if market['contracts']]

## Open Interest (must run chrome in debug mode)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [19]:
from OI_to_integer import OI_to_integer

In [20]:
from run_get_OI import run_get_OI

In [21]:
for market in tqdm(f2):
    for contract in tqdm(market['contracts']):
        if 'OI' not in contract:
            conid = contract['conid']
            OI = await run_get_OI([conid])
            OI = list(OI.values())[0]
            contract['OI'] = OI_to_integer(OI)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056496: 68.3K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056501: 68.3K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056506: 32.0K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056511: 32.0K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056520: 1.54M
Re

## Collect branches of single IFP of interest and construct IFP

In [22]:
rows = []
for market in f2:
    for ctx in market['contracts']:
        contract = ctx.copy()
        probability = contract['probability']
        contract['probability_pct'] = probability['probability_pct']
        contract['weekly_volume'] = sum(probability['volume'])
        contract['yesNo'] = 'YES' if contract['putOrCall'] == 'C' else 'NO'
        for x in ['putOrCall','popularityRank', 'expiration', 'lastTradeMillis', 'lastTradeTime','eventFixedPayout', 'commodityCode', 'strike', 'market',
                    'categories', 'expectedResolutionTime', 'expectedPayoutTime', 'timespecifierParam', 'sourceAgency', 'marketRulesLink', 
                    'exchange', 'priceIncrement', 'currency', 'timezone', 'eventAuthorityURL', 'probability', 'tradingHours']:
            del contract[x]
        contract['category1'] = market['category1']
        contract['category2'] = market['category2']
        contract['category3'] = market['category3']
        rows.append(contract)

In [23]:
df = pd.DataFrame(rows)

In [24]:
df1 = df[['strikeLabel', 'yesNo', 'probability_pct', 'weekly_volume', 'OI']]
df2 = df1[df1.yesNo == 'YES'][['strikeLabel','probability_pct', 'weekly_volume', 'OI']].copy()
df2['Flow (Volume as % of OI)'] = (100 * df2.weekly_volume/df2.OI).apply(int)
df2

,strikeLabel,probability_pct,weekly_volume,OI,Flow (Volume as % of OI)
0,Sliwa,4.0,130.0,68300.0,0
2,Cuomo,12.0,4600.0,32000.0,14
4,Mamdani,77.0,357967.0,1540000.0,23
6,Adams,10.0,6753.0,201000.0,3


In [25]:
bins = df[df.yesNo == 'YES'].apply(lambda row: f'{row.longDescription}', axis=1).values.tolist()

In [26]:
bins

['Will Curtis Sliwa win the New York City general election for mayor in 2025?',
 'Will Andrew Cuomo win the New York City general election for mayor in 2025?',
 'Will Zohran Mamdani win the New York City general election for mayor in 2025?',
 'Will Eric Adams win the New York City general election for mayor in 2025?']

In [27]:
bins = [{'props': {'title': bin, 'ai_title': '', 'order': 0, 'color': ''}, 'isActive': True, 'endDay': 0}
        for bin in bins]

In [28]:
from datetime import datetime

In [29]:
ifp = {'id': market['conid'],
       'category': 'Election',
        'type': 'numeric',
        'state': 'active',
        'dates': {'startDay': str(datetime.now())[0:10].replace('-',''),'endDay': contract['lastTradeDate']},
        'props': {'title': market['name'], 'shortTitle': '', 'details': ''},
        'kind': 'discrete',
        'bins': bins}

In [30]:
ifp

{'id': 796056051,
 'category': 'Election',
 'type': 'numeric',
 'state': 'active',
 'dates': {'startDay': '20250801', 'endDay': '20251129'},
 'props': {'title': 'General Election for New York City Mayor',
  'shortTitle': '',
  'details': ''},
 'kind': 'discrete',
 'bins': [{'props': {'title': 'Will Curtis Sliwa win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Andrew Cuomo win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Zohran Mamdani win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Eric Adams win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color':

In [31]:
ifps = [ifp]

## Gather news relevant to branches

In [32]:
from save_ifps_to_disk import save_ifps_to_disk
id_to_ifp = save_ifps_to_disk(ifps)

In [33]:
from detailed_proposition import detailed_proposition
from wiki_semantic_search import wiki_semantic_search
from split_news_into_text_and_urls import split_news_into_text_and_urls
from create_source_summaries import create_source_summaries
from rephrase_binary_outcomes import rephrase_binary_outcomes
from format_research import format_research
from glimt_forecast_prompt import glimt_forecast_prompt
from get_forecast_components import *
from median_forecast import median_forecast
from median_rationale import median_rationale
from datetime import datetime

loading massive wiki index 2025-08-01 12:22:01.051978
loading wiki article titles 2025-08-01 12:23:10.557610
loading sentence transformer model 2025-08-01 12:23:13.848307
done 2025-08-01 12:23:17.794073


In [34]:
for ifp in ifps:
    break

In [35]:
topics = [ifp['props']['title']] + [bin['props']['title'] for bin in ifp['bins']]

In [74]:
topics

['General Election for New York City Mayor',
 'Will Curtis Sliwa win the New York City general election for mayor in 2025?',
 'Will Andrew Cuomo win the New York City general election for mayor in 2025?',
 'Will Zohran Mamdani win the New York City general election for mayor in 2025?',
 'Will Eric Adams win the New York City general election for mayor in 2025?']

In [36]:
from call_asknews import call_asknews

In [37]:
news = [call_asknews(topic) for topic in topics]

In [38]:
news1 = [x.replace('Here are the relevant news articles:','').strip() for x in news]

In [ ]:
news2 = []
for block in news1:
    L = block.split('\n\n')
    news2.extend(L)
news2 = list(sorted(set(news2)))

In [103]:
research = '\n\n'.join(news2)

In [96]:
fn = f"glimt/news/{ifp['id']}.txt"
with open(fn, 'w') as f:
    f.write(news3)

In [101]:
### Split news into text and URLs
s2 = [x for x in news2 if '\nSource' in x]
s3 = [x.split('\nSource:') for x in s2]
ifp_news_text = [x[0].split('\nOriginal language:')[0] for x in s3]
ifp_news_sources = [x.split('](')[1][0:-1] for x in [x[1] for x in s3]]
ifp_news = dict([(x,y) for x,y in zip(ifp_news_sources, ifp_news_text)])
ifp_news_sources = [x for x,y in ifp_news.items()]
ifp_news_text = [y for x,y in ifp_news.items()]

## Forecast branches with advanced prompt engineering

### Overview

https://www.perplexity.ai/search/on-rtx-6000-ada-deepseek-r1-70-lE9ZMYNMQWKh.hNj5i73OQ?1=r#19

Below is a **Python script template** designed for your scenario: starting from only the election question name and bins, plus raw `research` and `wiki` texts, it constructs a staged **dialogue with the LLM**. This progressively:

- Extracts and verifies core facts about candidates and the race,
- Grounds analysis on explicit recent data,
- Applies stepwise reasoning to counter the common “incumbency bias”,
- Produces a calibrated probability distribution,
- Lists supporting and contradicting reasoning.

The script is **generic and robust to any Election question**, discovering candidates from input bins, using supplied texts, and never hardcoding assumptions.

- Uses **chained prompts** that feed the prior response into the next prompt context (dialogue style).
- Enforces explicit focus on research and wiki data, no priors.
- Explicitly extracts nominee status, polling, coalitions, and approval ratings per candidate.
- Commands the model to reason step-by-step avoiding default incumbent assumptions.
- Outputs probabilities and rationales in a strict format.


### Conversation history: store (user_prompt, model_response) tuples if your LLM supports dialogue history.

In [129]:
from call_llm import call_llm
conversation = []

### Step 1: Introduce the question and ask for nomination confirmation --

In [130]:
title = ifp['props']['title']

In [131]:
bins = topics[1:]
prompt_1 = f"""
You are a highly capable election analyst tasked with forecasting the outcome of the following election question:

```
{title}
```

Candidates/bins in this race are:

{', '.join(bins)}

You are given the following news about the election and candidates:

{research}

Your first task is:

1. For each candidate, determine from the given data whether they are officially
confirmed candidates as of the latest election stage.
2. State clearly which candidate(s) have won their party’s nomination or primary if that info is available.
3. Do not make assumptions beyond the given data; only use facts stated or strongly implied.

Provide your answer as a concise bullet list, one line per candidate, stating nomination status.

Example:
- Joe Blow: Republican nominee, confirmed.
- Dick Wix: Democratic nominee, confirmed.
- John Johnson: Independent candidate, confirmed.
- Mai Thang: Incumbent Mayor running independently.

Respond ONLY with that list.
"""

In [132]:
pdir = f"glimt/prompt/{ifp['id']}"

In [133]:
os.makedirs(pdir, exist_ok=True)

In [134]:
with open(f'{pdir}/prompt_1.txt', 'w') as f:
    f.write(prompt_1)

In [135]:
response_1 = call_llm(prompt_1, conversation)

In [136]:
confirmation_status = response_1.split('</think>')[1].strip()
print(confirmation_status)

- Zohran Mamdani: Democratic nominee, confirmed.  
- Andrew Cuomo: Lost Democratic primary, not confirmed.  
- Curtis Sliwa: Republican nominee, confirmed.  
- Eric Adams: Incumbent Mayor running independently, confirmed.  
- Jim Walden: Independent candidate, confirmed.


In [137]:
candidates = [x.split(':')[0] for x in confirmation_status.split('\n')]
candidates

['- Zohran Mamdani',
 '- Andrew Cuomo',
 '- Curtis Sliwa',
 '- Eric Adams',
 '- Jim Walden']

In [190]:
with open(f'{pdir}/confirmation_status.md', 'w') as f:
    f.write(confirmation_status)

### Step 2: Extract recent polling and support per candidate --

In [158]:
prompt_2 = f"""

You are given the following news about the election and candidates:

{research}

Based on the research and wiki data given before, extract the latest polling percentages or voter support
for each candidate:

{'\n'.join(candidates)}

If multiple polls exist, summarize their range or average.

Format the output as a bullet list, eg:

- Howdy Doody: 8-10%. SUPPORT
- Joe Blow: 6-7%. SUPPORT
- Doggy Woggy: 50-55%. SUPPORT 
- Pig Tong: 7-11%. SUPPORT

where the SUPPORT describes what polls gave you the range of numbers.
"""

In [159]:
with open(f'{pdir}/prompt_2.txt', 'w') as f:
    f.write(prompt_2)

In [160]:
response_2 = call_llm(prompt_2, conversation)

In [161]:
print("\nStep 2 polling data:\n", response_2)


Step 2 polling data:
 <think>
Okay, let's tackle this query. The user wants the latest polling percentages for each candidate in the NYC mayoral election based on the provided articles. First, I need to go through each of the sources mentioned and extract the relevant poll data.

Starting with Zohran Mamdani. The first article from July 14, 2025, by UrduPoint mentions a poll where he leads with 35%. Another article from July 30 by jta.org states he leads Jewish voters by 17 points, with 43% of Jewish voters. Then, the July 14 poll also mentions a margin of error of 4.2, so 35% could range from 30.8% to 39.2%. However, the July 30 article from PJ Media mentions a general election poll where he leads with 35%, but another article from July 14 gives him 35%, and the primary results from July 1 and July 2 show he won with 56%. So in the primary, he had 56%, but the general election polls vary. The July 14 poll in the general election gives him 35%, and the July 30 article (Zohran Mamdani 

In [173]:
polls = response_2.split('</think>')[1].strip()
print(polls)

- **Zohran Mamdani**: 30.8–39.2%.  
  *Support*: July 14 UrduPoint poll (35% ±4.2 MoE) and July 30 JTA poll (43% among Jewish voters, a subset).  

- **Andrew Cuomo**: 20.8–29.2%.  
  *Support*: July 14 UrduPoint poll (25% ±4.2 MoE) and July 30 JTA poll (26% among Jewish voters).  

- **Curtis Sliwa**: 9.8–18.2%.  
  *Support*: July 14 UrduPoint poll (14% ±4.2 MoE).  

- **Eric Adams**: 6.8–15.2%.  
  *Support*: July 14 UrduPoint poll (11% ±4.2 MoE).  

- **Jim Walden**: 0.8–5.2%.  
  *Support*: July 14 UrduPoint poll (1% ±4.2 MoE).  

*Note*: The July 14 UrduPoint poll (Slingshot Strategies) is the most comprehensive general election poll (sample size and methodology not specified). Other sources (e.g., Jewish voter subsets or primary results) are excluded for overall voter support.


In [189]:
with open(f'{pdir}/polls.md', 'w') as f:
    f.write(polls)

### Step 3: Compare polling data to incumbency bias --

In [176]:
prompt_3 = f"""
Bear in mind this research:

{research}

Using your findings from nominations and polling, analyze whether the data supports a typical "incumbent advantage"
for the incumbent candidate or if recent polling contradicts this.

Explicitly state if the incumbent is the frontrunner or if a newcomer leads.

Confirmation findings are:

{confirmation_status}

Polling findings are:

{polls}

Explain briefly in 3-5 sentences.
"""

In [177]:
with open(f'{pdir}/prompt_3.txt', 'w') as f:
    f.write(prompt_3)

In [178]:
response_3 = call_llm(prompt_3, conversation)
print("\nStep 3 incumbency analysis:\n", response_3)


Step 3 incumbency analysis:
 <think>
Okay, so I need to figure out if the data supports a typical "incumbent advantage" for Eric Adams or if recent polling contradicts that. Let's start by recalling what an incumbent advantage usually is. Typically, incumbents have an edge because they're current office holders, people know them, they have name recognition, and they can point to their record. But in this case, Eric Adams is the incumbent mayor, but he's running independently. That's a bit unusual.

Looking at the polling data from the July 14 UrduPoint poll, which is the most comprehensive one mentioned. Eric Adams has 11% support, while Zohran Mamdani, the Democratic nominee, has 35%. The other candidates are way behind. So, the incumbent isn't leading. That's a problem for the usual incumbent advantage. Usually, Adams should be higher, but he's not even in the top two. 

Also, Adams' net approval rating is -34, which is pretty low. That might explain why he's not doing well. The fac

In [179]:
incumbency_advantage = response_3.split('</think>')[1].strip()

In [180]:
print(incumbency_advantage)

The data contradicts a typical "incumbent advantage" for Eric Adams. While incumbents usually benefit from name recognition and institutional support, Adams (running independently) trails significantly in polling, with 11% support compared to Zohran Mamdani’s 35% (July 14 UrduPoint poll). Adams’ net approval rating (-34) and his decision to run outside the Democratic Party likely weakened his position, allowing Mamdani, a newcomer, to lead decisively. Thus, the incumbent is not the frontrunner; Mamdani, the Democratic nominee, holds a clear lead.


In [188]:
with open(f'{pdir}/incumbency_advantage.md', 'w') as f:
    f.write(incumbency_advantage)

### Step 4: Analyze coalitions, endorsements, and voter demographics per candidate --

In [182]:
prompt_4 = f"""

Bear in mind this research:

{research}

Confirmation findings are:

{confirmation_status}

Polling findings are:

{polls}

Incumbency advantage analysis concludes:

{incumbency_advantage}

Using the above information, analyze coalition strengths, key endorsements, and voter demographic
influences for each candidate.  Note how these factors might boost or reduce each candidate’s chances compared to raw polling numbers.
Explain briefly with references to the data you see.
"""

In [183]:
with open(f'{pdir}/prompt_4.txt', 'w') as f:
    f.write(prompt_4)

In [184]:
response_4 = call_llm(prompt_4, conversation)
print("\nStep 4 coalitions & endorsements:\n", response_4)


Step 4 coalitions & endorsements:
 <think>
Okay, let's tackle this analysis. So the user wants me to look at the coalition strengths, key endorsements, and voter demographics for each candidate in the NYC mayoral race. They also want to know how these factors might affect each candidate's chances compared to their raw polling numbers. Alright, let's start by going through each candidate one by one.

First, Zohran Mamdani. He's the Democratic nominee and has a strong lead in the polls. The data shows he's leading with 35% in the July 14 poll and even more among Jewish voters. His coalition probably includes progressive Democrats, young voters, and maybe some Jewish voters despite his criticism of Israel. The JTA poll mentions he's leading Jewish voters by 17 points, which is significant. His endorsements might come from progressive groups and possibly some union leaders. His platform includes things like zero-fare transit and rent freezes, which would appeal to lower-income voters and 

In [185]:
coalition_analysis = response_4.split('</think>')[1].strip()

In [187]:
with open(f'{pdir}/coalition_analysis.md', 'w') as f:
    f.write(coalition_analysis)

### Step 5: Request final probability forecast

In [193]:
sbins = '\n'.join(bins)

In [214]:
probs = '\n'.join([f"* {bin}: P{i}" for i, bin in enumerate(bins)])

In [215]:
# Prepare a minimal summary for model to generate probabilities
sbins = "\n".join([f"* O{i+1}. Will {c} win?" for i, c in enumerate(bins)])

prompt_5 = f"""
Based on this analysis:

{coalition_analysis}

You are asked to provide an election forecast with probabilities based on ALL prior analysis.

- Possible outcomes are:
{sbins}

- Probabilities Pi must satisfy 0 <= Pi <= 1 and sum to 1.0.
- Provide reasons for each probability as short text, linked to polling, coalitions, etc.
- At the end output the following:

```forecast
{probs}
```

Do NOT include code comments or explanations.
"""

In [216]:
with open(f'{pdir}/prompt_5.txt', 'w') as f:
    f.write(prompt_5)

In [217]:
response_5 = call_llm(prompt_5, conversation)
print("\nStep 5 probability forecast:\n", response_5)


Step 5 probability forecast:
 <think>
Okay, let's tackle this. The user wants me to generate a forecast based on the provided analysis. The previous answer had probabilities for the four candidates, but the user now wants the outcomes phrased as "Will [Candidate] win...?" with the same probabilities. I need to make sure the probabilities are correctly assigned to each outcome as per the analysis.

Looking at the analysis, Zohran Mamdani is the strong favorite with a 58% chance. Curtis Sliwa, the Republican nominee, has 24% due to his Trump base and anti-crime appeal but limited by NYC's Democratic lean. Eric Adams, the incumbent, has 13% because of his low approval and fragmented coalition. Andrew Cuomo is at 5% because he's a former governor who lost the primary, with weak support now.

I need to map each "Will [Candidate] win..." question to the correct probability. The user's example output uses P0 to P3, but the actual labels should correspond to each candidate. Let me check the o

In [225]:
binProbs = response_5.split('```forecast')[1].replace('```', '').strip().split('\n')
binProbs = [float(x.split(': ')[1]) for x in binProbs]
binProbs

[0.24, 0.05, 0.58, 0.13]

In [226]:
with open(f'{pdir}/binProbs.txt', 'w') as f:
    f.write(str(binProbs))

In [207]:
with open(f'{pdir}/rationale.txt', 'w') as f:
    f.write(rationale)

### Step 6: Request reasons why forecast might be right --

In [234]:
reasons = '\n'.join([f"* Reason for {bin[:-1]}: <rationale here>. <Ending with forecast ZZ% probability>." for i, bin in enumerate(bins)])

In [235]:
print(reasons)

* Reason for Will Curtis Sliwa win the New York City general election for mayor in 2025: <rationale here>. <Ending with forecast ZZ% probability>.
* Reason for Will Andrew Cuomo win the New York City general election for mayor in 2025: <rationale here>. <Ending with forecast ZZ% probability>.
* Reason for Will Zohran Mamdani win the New York City general election for mayor in 2025: <rationale here>. <Ending with forecast ZZ% probability>.
* Reason for Will Eric Adams win the New York City general election for mayor in 2025: <rationale here>. <Ending with forecast ZZ% probability>.


In [239]:
prompt_6 = f"""
For each candidate outcome, give a concise reason why your forecasted probability might be correct,
grounded in current data and reasoning.  Your rationale is

{rationale}

to assign probabilities

{binProbs}

to candidates.

Wrap your reasons in the markdown tag like this:

```
{reasons}
```
"""

In [240]:
with open(f'{pdir}/prompt_6.txt', 'w') as f:
    f.write(prompt_6)

In [241]:
response_6 = call_llm(prompt_6, conversation)
print("\nStep 6 reasons forecast is right:\n", response_6)


Step 6 reasons forecast is right:
 <think>
Okay, I need to create a rationale for each candidate based on the given analysis and the assigned probabilities. Let me start by reviewing the candidate analyses provided.

Zohran Mamdani is the Democratic nominee with strong progressive support, a solid Jewish voter lead, and a 6:1 Democratic registration edge. His polling numbers might be underestimated because the UrduPoint poll shows 35%, but his coalition is robust. So his probability is highest at 58%.

Curtis Sliwa is the Republican nominee with Trump-aligned voters and anti-crime supporters. The analysis mentions that his 14% in polls is underestimated but NYC's Democratic lean limits his ceiling. The probability is 24%, which accounts for some Trump voters and anti-crime independents but not enough to overcome the Democratic majority.

Andrew Cuomo lost the primary and has limited support. His 25% in the poll is likely inflated. His coalition is fractured, and he lacks party backing

In [242]:
rRight = response_6.split('```')[1].split('```')[0]

In [243]:
print(rRight)


* Reason for Will Curtis Sliwa win the New York City general election for mayor in 2025: Sliwa’s coalition of Trump-aligned voters (30% of 2024 supporters) and anti-crime advocates could marginally elevate his support beyond current 14% (UrduPoint), but NYC’s 6:1 Democratic registration and weak Republican mobilization cap his ceiling. <Ending with forecast 24% probability>.
* Reason for Will Andrew Cuomo win the New York City general election for mayor in 2025: Cuomo’s fractured coalition of moderate suburbanites (25% in UrduPoint) and older Jewish voters (26% in JTA) lacks party backing and faces Democratic backlash. His low polling reflects a weak base in a city dominated by progressive Democrats. <Ending with forecast 5% probability>.
* Reason for Will Zohran Mamdani win the New York City general election for mayor in 2025: Mamdani’s coalition—progressives (67% under-44 Jewish support, per JTA), youth, and a 43% Jewish voter lead—combined with Democratic unity (6:1 registration ad

In [244]:
with open(f'{pdir}/rRight.md', 'w') as f:
    f.write(rRight)

### Step 7: Request reasons forecast might be wrong --

In [245]:
wrongs = '\n'.join([f"* Regarding forecast for {bin[:-1]}: <Reasons the forecast might be wrong>." for i, bin in enumerate(bins)])

In [247]:
prompt_7 = f"""
Your rationale is

{rationale}

to assign probabilities

{binProbs}

to candidates.

For each candidate outcome, provide plausible reasons why your forecast might be wrong,
focusing on uncertainties like polling errors, turnout changes, late developments,
or unexpected events.

Wrap your reasons you might be wrong in the markdown tag like this:

```
{wrongs}
```
"""

In [248]:
with open(f'{pdir}/prompt_7.txt', 'w') as f:
    f.write(prompt_7)

In [250]:
response_7 = call_llm(prompt_7, conversation)
print("\nStep 7 reasons forecast might be wrong:\n", response_7)


Step 7 reasons forecast might be wrong:
 <think>
Okay, let's see. The user wants me to think through the possible reasons each forecast might be wrong. They mentioned uncertainties like polling errors, turnout changes, late developments, or unexpected events.

Starting with Curtis Sliwa. His forecast is 24%. The original analysis said his support might be underestimated because he could get Trump voters and anti-crime independents. But what if the polls are right? If he doesn't actually get that Trump support, maybe because Trump voters are not as numerous as expected, or if the anti-crime issue doesn't resonate as much. Also, if Democratic turnout is super high, that could crush his chances. Maybe a surge in Democratic enthusiasm could make his 24% too high.

Andrew Cuomo's forecast is 5%. The analysis says his coalition is weak, but if there's a late surge from moderate voters, especially if Mamdani's policies are seen as too extreme. Also, if there's a scandal or some event that ma

In [251]:
rWrong = response_7.split('```')[1].split('```')[0]

In [252]:
with open(f'{pdir}/rWrong.md', 'w') as f:
    f.write(rWrong)

### Final output

In [256]:
results = {
    "nominations": confirmation_status,
    "polling": polls,
    "incumbency_analysis": incumbency_advantage,
    "coalitions": coalition_analysis,
    "probabilities": probs,
    "reasons_right": rRight,
    "reasons_wrong": rWrong,
}

In [257]:
for key in results:
    ifp[key] = results[key]

In [258]:
ifp

{'id': 796056051,
 'category': 'Election',
 'type': 'numeric',
 'state': 'active',
 'dates': {'startDay': '20250801', 'endDay': '20251129'},
 'props': {'title': 'General Election for New York City Mayor',
  'shortTitle': '',
  'details': ''},
 'kind': 'discrete',
 'bins': [{'props': {'title': 'Will Curtis Sliwa win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Andrew Cuomo win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Zohran Mamdani win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Eric Adams win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color':